# emotion2vec VADヘッド 動作確認

wav2vec 2.0 の設計に倣い、事前学習済み音声表現の上に軽量な下流ヘッドを追加して VAD（Valence / Arousal / Dominance）を出力するための確認ノートブックです。

- wav2vec 2.0: raw waveform → CNN feature encoder → Transformer context network → task head
- このノートブック: emotion2vec features `(B, T, 768)` → Attention pooling → FNN VAD head → optional emotion classifier
- 目的: モデル構造、padding mask、出力範囲、Stage 1/2 の勾配、簡易 overfit の動きを確認する

## 設計対応

wav2vec 2.0 論文では、自己教師あり事前学習で得た contextual representation を、ラベル付きデータで下流タスクに fine-tune します。ASR では CTC 用の出力層を追加します。

ここでは同じ考え方で、emotion2vec の contextual frame representation に VAD 回帰ヘッドを追加します。VAD は発話単位の連続値として扱うため、Transformer 出力列を padding-aware attention pooling で集約してから `[-1, 1]` の3次元へ写像します。

参照: [wav2vec 2.0 arXiv:2006.11477](https://arxiv.org/abs/2006.11477), [NeurIPS 2020 paper PDF](https://papers.nips.cc/paper_files/paper/2020/file/92d1e1eb1cd6f9fba3227870bb6d7f07-Paper.pdf)

In [ ]:
# このセルでは、プロジェクトパスを追加し、VADモデル検証に必要なライブラリを読み込む。

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn, optim

ROOT = Path.cwd()
if ROOT.name == 'vad_downstream':
    ROOT = ROOT.parent

VAD_DIR = ROOT / 'vad_downstream'
if str(VAD_DIR) not in sys.path:
    sys.path.insert(0, str(VAD_DIR))

from model import AttentionPooling, VADDecoder, EmotionClassifier
from loss import ccc_loss, stage1_loss

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'ROOT   : {ROOT}')
print(f'DEVICE : {DEVICE}')
print(f'torch  : {torch.__version__}')

In [ ]:
# このセルでは、検証用のEmotionClassifierを作成し、パラメータ数とデバイスを確認する。

INPUT_DIM = 768
HIDDEN_DIM = 256
VAD_DIM = 3
NUM_CLASSES = 4

model = EmotionClassifier(
    input_dim=INPUT_DIM,
    hidden_dim=HIDDEN_DIM,
    vad_dim=VAD_DIM,
    num_classes=NUM_CLASSES,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model)
print(f'parameters: {n_params:,} / trainable: {n_trainable:,}')

In [ ]:
# このセルでは、長さの異なるダミー特徴量を作り、forward出力のshapeを確認する。

lengths = torch.tensor([120, 80, 17], device=DEVICE)
batch_size = len(lengths)
max_len = int(lengths.max().item())

feats = torch.randn(batch_size, max_len, INPUT_DIM, device=DEVICE)
padding_mask = torch.arange(max_len, device=DEVICE).unsqueeze(0) >= lengths.unsqueeze(1)

vad, logits = model(feats, padding_mask)

assert vad.shape == (batch_size, VAD_DIM)
assert logits.shape == (batch_size, NUM_CLASSES)
assert torch.all(vad <= 1.0 + 1e-6) and torch.all(vad >= -1.0 - 1e-6)

print(f'input feats  : {tuple(feats.shape)}')
print(f'padding mask : {tuple(padding_mask.shape)} / padded frames={int(padding_mask.sum())}')
print(f'VAD output   : {tuple(vad.shape)}')
print(f'logits       : {tuple(logits.shape)}')
print(vad.detach().cpu())

In [ ]:
# このセルでは、AttentionPoolingがpadding位置を無視して重み付けしているか確認する。

model.eval()
with torch.no_grad():
    scores = model.vad_decoder.pool.score(feats).squeeze(-1)
    masked_scores = scores.masked_fill(padding_mask, float('-inf'))
    weights = torch.softmax(masked_scores, dim=-1)

weight_sums = weights.sum(dim=-1)
padded_weight_sum = (weights * padding_mask.float()).sum(dim=-1)

print('attention weight sums       :', weight_sums.detach().cpu().numpy())
print('padded-frame weight sums   :', padded_weight_sum.detach().cpu().numpy())

assert torch.allclose(weight_sums, torch.ones_like(weight_sums), atol=1e-6)
assert torch.all(padded_weight_sum < 1e-6)

plt.figure(figsize=(8, 2.4))
plt.plot(weights[1].detach().cpu().numpy())
plt.axvline(int(lengths[1].item()), color='red', linestyle='--', linewidth=1)
plt.title('Attention weights for sample 1')
plt.xlabel('frame')
plt.ylabel('weight')
plt.tight_layout()

## Stage 1: VA 連続値で VAD ヘッドを学習

Dominance ラベルがない想定では、`stage1_loss` は Valence と Arousal の CCC loss だけを使います。分類器は forward されますが、この損失からは勾配を受けません。

In [ ]:
# このセルでは、Stage 1のVA回帰損失でVADDecoderに勾配が流れるか確認する。

model.train()
model.zero_grad(set_to_none=True)

va_target = torch.empty(batch_size, 2, device=DEVICE).uniform_(-1.0, 1.0)
vad, _ = model(feats, padding_mask)
loss_s1 = stage1_loss(vad, va_target)
loss_s1.backward()

vad_grad = sum(
    p.grad.detach().abs().sum().item()
    for p in model.vad_decoder.parameters()
    if p.grad is not None
)
cls_grad = sum(
    p.grad.detach().abs().sum().item()
    for p in model.classifier.parameters()
    if p.grad is not None
)

print(f'Stage 1 loss         : {loss_s1.item():.6f}')
print(f'VAD decoder grad sum : {vad_grad:.6f}')
print(f'classifier grad sum  : {cls_grad:.6f}')

assert vad_grad > 0
assert cls_grad == 0

## Stage 2: 感情分類で VAD 空間を調整

Stage 2 では `Linear(3 → num_classes)` の分類器を CrossEntropy で学習します。VAD ヘッドも小さい学習率で更新すると、VAD 空間を分類に合わせて微調整できます。

In [ ]:
# このセルでは、Stage 2の分類損失で分類器とVADDecoderに勾配が流れるか確認する。

model.train()
model.zero_grad(set_to_none=True)

labels = torch.tensor([0, 1, 2], dtype=torch.long, device=DEVICE)
criterion = nn.CrossEntropyLoss()

_, logits = model(feats, padding_mask)
loss_s2 = criterion(logits, labels)
loss_s2.backward()

vad_grad = sum(
    p.grad.detach().abs().sum().item()
    for p in model.vad_decoder.parameters()
    if p.grad is not None
)
cls_grad = sum(
    p.grad.detach().abs().sum().item()
    for p in model.classifier.parameters()
    if p.grad is not None
)

print(f'Stage 2 loss         : {loss_s2.item():.6f}')
print(f'VAD decoder grad sum : {vad_grad:.6f}')
print(f'classifier grad sum  : {cls_grad:.6f}')

assert vad_grad > 0
assert cls_grad > 0

In [ ]:
# このセルでは、小さな合成データに過学習できるかを試し、モデル実装の健全性を確認する。

toy_model = EmotionClassifier(
    input_dim=INPUT_DIM,
    hidden_dim=128,
    vad_dim=VAD_DIM,
    num_classes=NUM_CLASSES,
).to(DEVICE)

toy_batch = 16
toy_len = 48
toy_feats = torch.randn(toy_batch, toy_len, INPUT_DIM, device=DEVICE)
toy_mask = torch.zeros(toy_batch, toy_len, dtype=torch.bool, device=DEVICE)

with torch.no_grad():
    pooled = toy_feats[:, :, :2].mean(dim=1)
    toy_va = torch.tanh(2.0 * pooled)

optimizer = optim.Adam(toy_model.vad_decoder.parameters(), lr=3e-3)
losses = []

for step in range(80):
    optimizer.zero_grad(set_to_none=True)
    pred_vad, _ = toy_model(toy_feats, toy_mask)
    loss = stage1_loss(pred_vad, toy_va)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

print(f'first loss: {losses[0]:.4f}')
print(f'last loss : {losses[-1]:.4f}')

plt.figure(figsize=(6, 3))
plt.plot(losses)
plt.title('Toy Stage 1 overfit check')
plt.xlabel('step')
plt.ylabel('CCC loss')
plt.tight_layout()

In [ ]:
# このセルでは、合成データ上のVAD予測を可視化し、出力レンジと傾向を確認する。

toy_model.eval()
with torch.no_grad():
    pred_vad, _ = toy_model(toy_feats, toy_mask)

pred = pred_vad.detach().cpu().numpy()
target = toy_va.detach().cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(8, 3), sharex=True, sharey=True)
axes[0].scatter(target[:, 0], target[:, 1], c='tab:blue')
axes[0].set_title('target VA')
axes[1].scatter(pred[:, 0], pred[:, 1], c='tab:orange')
axes[1].set_title('predicted VA')
for ax in axes:
    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)
    ax.set_xlabel('Valence')
    ax.set_ylabel('Arousal')
    ax.grid(True, alpha=0.3)
plt.tight_layout()

print('predicted VAD sample:')
print(np.round(pred[:5], 3))

## 任意: 抽出済み emotion2vec 特徴で1バッチ確認

`FEATURE_BASE` に拡張子なしの特徴ファイルパスを入れると、`<base>.npy`, `<base>.lengths`, `<base>.emo` と `VA_PATH` を読み込んで実データの1バッチを通します。未設定のままならスキップします。

In [ ]:
# このセルでは、実際のemotion2vec特徴量とVAラベルを読み込み、1バッチを確認する。

FEATURE_BASE = ''  # 例: r'C:/path/to/train'  -> train.npy, train.lengths, train.emo
VA_PATH = ''       # 例: r'C:/path/to/va_labels.txt'

if FEATURE_BASE and VA_PATH:
    from data import build_dataloaders, load_iemocap_with_va

    label_dict = {'ang': 0, 'hap': 1, 'neu': 2, 'sad': 3}
    real_data = load_iemocap_with_va(FEATURE_BASE, label_dict, VA_PATH)
    train_loader, val_loader, test_loader = build_dataloaders(
        real_data,
        batch_size=4,
        test_start=0,
        test_end=min(4, real_data['num']),
        eval_is_test=True,
    )
    batch = next(iter(train_loader))
    real_feats = batch['net_input']['feats'].to(DEVICE)
    real_mask = batch['net_input']['padding_mask'].to(DEVICE)
    with torch.no_grad():
        real_vad, real_logits = model(real_feats, real_mask)
    print(f'real feats : {tuple(real_feats.shape)}')
    print(f'real VAD   : {tuple(real_vad.shape)}')
    print(f'real logits: {tuple(real_logits.shape)}')
else:
    print('FEATURE_BASE / VA_PATH が未設定なのでスキップしました。')

## 任意: raw WAV → emotion2vec → VAD

ローカルに emotion2vec の fairseq checkpoint がある場合だけ実行します。`CHECKPOINT_PATH` を設定すると、`scripts/test.wav` から frame features を抽出して VAD ヘッドへ渡します。

In [ ]:
# このセルでは、wavからemotion2vec特徴量を抽出してVAD推定まで通す手順を確認する。

CHECKPOINT_PATH = ''  # 例: r'C:/path/to/emotion2vec_base.pt'
WAV_PATH = ROOT / 'scripts' / 'test.wav'

if CHECKPOINT_PATH:
    from dataclasses import dataclass
    import soundfile as sf
    import torch.nn.functional as F
    import fairseq

    @dataclass
    class UserDirModule:
        user_dir: str

    fairseq.utils.import_user_module(UserDirModule(str(ROOT / 'upstream')))
    ensemble, cfg, task = fairseq.checkpoint_utils.load_model_ensemble_and_task([CHECKPOINT_PATH])
    e2v = ensemble[0].to(DEVICE).eval()

    wav, sr = sf.read(str(WAV_PATH), dtype='float32')
    assert sr == 16000, f'expected 16 kHz, got {sr}'
    if wav.ndim == 2:
        wav = wav.mean(axis=1)

    source = torch.from_numpy(wav).to(DEVICE)
    if task.cfg.normalize:
        source = F.layer_norm(source, source.shape)
    source = source.view(1, -1)

    with torch.no_grad():
        extracted = e2v.extract_features(source, padding_mask=None, mask=False)
        frame_feats = extracted['x']
        frame_mask = extracted['padding_mask']
        if frame_mask is None:
            frame_mask = torch.zeros(frame_feats.shape[:2], dtype=torch.bool, device=DEVICE)
        wav_vad, wav_logits = model(frame_feats, frame_mask)

    print(f'wav samples : {tuple(source.shape)}')
    print(f'features    : {tuple(frame_feats.shape)}')
    print(f'VAD         : {wav_vad.detach().cpu().numpy().round(3)}')
    print(f'logits      : {wav_logits.detach().cpu().numpy().round(3)}')
else:
    print('CHECKPOINT_PATH が未設定なのでスキップしました。')

## 次に実データで確認する点

1. VA ラベルがある場合: Stage 1 の CCC loss が下がるか確認する。
2. 感情カテゴリがある場合: Stage 2 の WA / UA / F1 を確認する。
3. Dominance ラベルがない場合: D 次元は分類損失から間接的に動くため、分類器 `Linear(3 → N)` の D 列重みや、V/A 散布図とクラス分離を併せて見る。